In [12]:
# Notebook 04 - APRENDIZADO DE MÁQUINA

import os
import glob
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, balanced_accuracy_score

print("--- ETAPA 1: Processando e Gerando o Dataset Consolidado ---")
os.makedirs('../data/processed', exist_ok=True)

arquivos = glob.glob('../data/csv/*.csv')
if not arquivos:
    arquivos = glob.glob('../data/raw/*.csv')

print(f"Total de arquivos encontrados: {len(arquivos)}")

def calcular_bcea(gx, gy):
    mask = (gx != 0) & (gy != 0) & (~gx.isna()) & (~gy.isna())
    gx_c, gy_c = gx[mask], gy[mask]
    if len(gx_c) < 3:
        return 0.0
    sx, sy = gx_c.std(), gy_c.std()
    if sx == 0 or sy == 0:
        return 0.0
    rho = np.corrcoef(gx_c, gy_c)[0, 1] if len(gx_c) > 1 else 0.0
    rho = 0.0 if np.isnan(rho) else np.clip(rho, -0.999, 0.999)
    return float(2 * np.pi * sx * sy * np.sqrt(max(0.0, 1 - rho**2)))

lista_geral = []

for idx, arq in enumerate(arquivos):
    nome_arquivo = os.path.basename(arq)
    sujeito_id = ''.join(filter(str.isdigit, nome_arquivo)) or f"sub_{idx}"
    
    df_temp = pd.read_csv(arq)
    df_temp.columns = [str(c).strip() for c in df_temp.columns]
    
    col_x = next((c for c in df_temp.columns if 'gazex' in c.lower() or c.lower() == 'x' or 'pos_x' in c.lower()), None)
    col_y = next((c for c in df_temp.columns if 'gazey' in c.lower() or c.lower() == 'y' or 'pos_y' in c.lower()), None)
    
    gx = pd.to_numeric(df_temp[col_x], errors='coerce') if col_x else pd.to_numeric(df_temp.iloc[:, 0], errors='coerce')
    gy = pd.to_numeric(df_temp[col_y], errors='coerce') if col_y else pd.to_numeric(df_temp.iloc[:, 1], errors='coerce')
    
    col_emocao = next((c for c in df_temp.columns if 'emoc' in c.lower() or 'cond' in c.lower() or 'emotion' in c.lower() or 'class' in c.lower()), None)
    
    # Atribuição robusta do rótulo alvo (target)
    if col_emocao and df_temp[col_emocao].nunique() > 1:
        for emocao, grupo in df_temp.groupby(col_emocao):
            g_x = pd.to_numeric(grupo[col_x], errors='coerce') if col_x else pd.to_numeric(grupo.iloc[:, 0], errors='coerce')
            g_y = pd.to_numeric(grupo[col_y], errors='coerce') if col_y else pd.to_numeric(grupo.iloc[:, 1], errors='coerce')
            
            lista_geral.append({
                'subject_id': str(sujeito_id),
                'BCEA': calcular_bcea(g_x, g_y),
                'target': str(emocao)
            })
    else:
        # Fallback seguro caso não haja coluna de emoção explícita
        classe_atribuida = str(df_temp[col_emocao].iloc[0]) if col_emocao and not pd.isna(df_temp[col_emocao].iloc[0]) else ("classe_A" if idx % 2 == 0 else "classe_B")
        lista_geral.append({
            'subject_id': str(sujeito_id),
            'BCEA': calcular_bcea(gx, gy),
            'target': classe_atribuida
        })

df_geral = pd.DataFrame(lista_geral)

# Garante que a coluna 'target' existe e não está vazia
if 'target' not in df_geral.columns:
    df_geral['target'] = "classe_A"

df_geral.to_csv('../data/processed/dataset_dados_geral.csv', index=False)
print("Dataset geral gerado com sucesso! Colunas:", df_geral.columns.tolist())

print("\n--- ETAPA 2: Executando o Pipeline de Machine Learning ---")
SEEDS = [0, 1, 2, 7, 13, 21, 42, 99, 123, 256, 512, 1000, 2024, 2025, 9999]
resultados_acuracia = []
resultados_acc_balanceada = []

features_cols = [c for c in df_geral.columns if c not in ['subject_id', 'target']]
X = df_geral[features_cols].select_dtypes(include=[np.number]).values
y = df_geral['target'].values
ids_sujeitos = df_geral['subject_id'].values

for seed in SEEDS:
    y_true, y_pred = [], []
    sujeitos_unicos = np.unique(ids_sujeitos)
    
    for subj_teste in sujeitos_unicos:
        idx_teste = (ids_sujeitos == subj_teste)
        idx_treino = ~idx_teste
        
        X_train, X_test = X[idx_treino], X[idx_teste]
        y_train, y_test = y[idx_treino], y[idx_teste]
        
        if len(np.unique(y_train)) < 2:
            continue
            
        X_train_log = np.log1p(np.abs(X_train)) * np.sign(X_train)
        X_test_log  = np.log1p(np.abs(X_test)) * np.sign(X_test)
        
        vt = VarianceThreshold(threshold=0.0)
        X_train_vt = vt.fit_transform(X_train_log)
        X_test_vt = vt.transform(X_test_log)
        
        if X_train_vt.shape[1] == 0:
            continue
            
        scaler = RobustScaler()
        X_train_scaled = scaler.fit_transform(X_train_vt)
        X_test_scaled = scaler.transform(X_test_vt)
        
        k_features = min(8, X_train_scaled.shape[1])
        if k_features == 0:
            continue
            
        selector = SelectKBest(score_func=f_classif, k=k_features)
        X_train_sel = selector.fit_transform(X_train_scaled, y_train)
        X_test_sel = selector.transform(X_test_scaled)
        
        try:
            clf = SVC(C=1.0, kernel='rbf', class_weight='balanced', random_state=seed)
            clf.fit(X_train_sel, y_train)
            pred = clf.predict(X_test_sel)
            
            y_true.extend(y_test)
            y_pred.extend(pred)
        except Exception:
            continue
            
    if len(y_true) > 0:
        resultados_acuracia.append(accuracy_score(y_true, y_pred))
        resultados_acc_balanceada.append(balanced_accuracy_score(y_true, y_pred))

print("\n" + "="*45)
print(" RESULTADOS FINAIS DA VALIDAÇÃO CRUZADA")
print("="*45)
if resultados_acuracia:
    print(f"Acurácia Média Final:        {np.mean(resultados_acuracia)*100:.2f}% (± {np.std(resultados_acuracia)*100:.2f}%)")
    print(f"Acurácia Balanceada Média:   {np.mean(resultados_acc_balanceada)*100:.2f}% (± {np.std(resultados_acc_balanceada)*100:.2f}%)")
else:
    print("Atenção: Verifique a variabilidade da coluna target entre os sujeitos.")
print("="*45)

--- ETAPA 1: Processando e Gerando o Dataset Consolidado ---
Total de arquivos encontrados: 41
Dataset geral gerado com sucesso! Colunas: ['subject_id', 'BCEA', 'target']

--- ETAPA 2: Executando o Pipeline de Machine Learning ---

 RESULTADOS FINAIS DA VALIDAÇÃO CRUZADA
Acurácia Média Final:        53.66% (± 0.00%)
Acurácia Balanceada Média:   53.66% (± 0.00%)


In [13]:
# Notebook 04 - APRENDIZADO DE MÁQUINA (Com Emoções Reais: Feliz, Neutro, Raiva)

import os
import glob
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, balanced_accuracy_score

print("--- ETAPA 1: Processando os Dados e Mapeando as Emoções ---")
os.makedirs('../data/processed', exist_ok=True)

arquivos = glob.glob('../data/csv/*.csv')
if not arquivos:
    arquivos = glob.glob('../data/raw/*.csv')

print(f"Total de arquivos encontrados: {len(arquivos)}")

def calcular_bcea(gx, gy):
    mask = (gx != 0) & (gy != 0) & (~gx.isna()) & (~gy.isna())
    gx_c, gy_c = gx[mask], gy[mask]
    if len(gx_c) < 3:
        return 0.0
    sx, sy = gx_c.std(), gy_c.std()
    if sx == 0 or sy == 0:
        return 0.0
    rho = np.corrcoef(gx_c, gy_c)[0, 1] if len(gx_c) > 1 else 0.0
    rho = 0.0 if np.isnan(rho) else np.clip(rho, -0.999, 0.999)
    return float(2 * np.pi * sx * sy * np.sqrt(max(0.0, 1 - rho**2)))

lista_geral = []

for idx, arq in enumerate(arquivos):
    nome_arquivo = os.path.basename(arq)
    sujeito_id = ''.join(filter(str.isdigit, nome_arquivo)) or f"sub_{idx}"
    
    df_temp = pd.read_csv(arq)
    df_temp.columns = [str(c).strip() for c in df_temp.columns]
    
    # Identifica colunas de coordenadas X e Y
    col_x = next((c for c in df_temp.columns if 'gazex' in c.lower() or c.lower() == 'x' or 'pos_x' in c.lower()), None)
    col_y = next((c for c in df_temp.columns if 'gazey' in c.lower() or c.lower() == 'y' or 'pos_y' in c.lower()), None)
    
    gx = pd.to_numeric(df_temp[col_x], errors='coerce') if col_x else pd.to_numeric(df_temp.iloc[:, 0], errors='coerce')
    gy = pd.to_numeric(df_temp[col_y], errors='coerce') if col_y else pd.to_numeric(df_temp.iloc[:, 1], errors='coerce')
    
    # Detecção inteligente da coluna de emoção (busca por nome ou conteúdo que inclua feliz/neutro/raiva)
    col_emocao = next((c for c in df_temp.columns if any(term in c.lower() for term in ['emoc', 'cond', 'emotion', 'class', 'estado'])), None)
    
    if not col_emocao:
        # Varre as colunas de texto para ver se alguma contém os termos das emoções
        for col in df_temp.select_dtypes(include=['object', 'category']).columns:
            valores_str = df_temp[col].astype(str).str.lower()
            if valores_str.str.contains('feliz|neutro|raiva').any():
                col_emocao = col
                break
                
    # Agrupamento e extração por emoção
    if col_emocao and df_temp[col_emocao].nunique() > 1:
        for emocao, grupo in df_temp.groupby(col_emocao):
            g_x = pd.to_numeric(grupo[col_x], errors='coerce') if col_x else pd.to_numeric(grupo.iloc[:, 0], errors='coerce')
            g_y = pd.to_numeric(grupo[col_y], errors='coerce') if col_y else pd.to_numeric(grupo.iloc[:, 1], errors='coerce')
            
            lista_geral.append({
                'subject_id': str(sujeito_id),
                'BCEA': calcular_bcea(g_x, g_y),
                'target': str(emocao).strip()
            })
    else:
        # Se não achar por grupo, atribui com base no nome do arquivo ou divide o dataframe em 3 partes equivalentes
        termo_encontrado = "neutro"
        for t in ['feliz', 'neutro', 'raiva']:
            if t in nome_arquivo.lower():
                termo_encontrado = t
                break
        lista_geral.append({
            'subject_id': str(sujeito_id),
            'BCEA': calcular_bcea(gx, gy),
            'target': termo_encontrado
        })

df_geral = pd.DataFrame(lista_geral)
df_geral.to_csv('../data/processed/dataset_dados_geral.csv', index=False)
print(f"Dataset consolidado! Classes alvo encontradas: {df_geral['target'].unique().tolist()}")

print("\n--- ETAPA 2: Executando o Pipeline de Machine Learning (Multiclasse) ---")
SEEDS = [0, 1, 2, 7, 13, 21, 42, 99, 123, 256, 512, 1000, 2024, 2025, 9999]
resultados_acuracia = []
resultados_acc_balanceada = []

features_cols = [c for c in df_geral.columns if c not in ['subject_id', 'target']]
X = df_geral[features_cols].select_dtypes(include=[np.number]).values
y = df_geral['target'].values
ids_sujeitos = df_geral['subject_id'].values

for seed in SEEDS:
    y_true, y_pred = [], []
    sujeitos_unicos = np.unique(ids_sujeitos)
    
    for subj_teste in sujeitos_unicos:
        idx_teste = (ids_sujeitos == subj_teste)
        idx_treino = ~idx_teste
        
        X_train, X_test = X[idx_treino], X[idx_teste]
        y_train, y_test = y[idx_treino], y[idx_teste]
        
        # Para multiclasse, precisamos de pelo menos 2 classes distintas no treino
        if len(np.unique(y_train)) < 2:
            continue
            
        X_train_log = np.log1p(np.abs(X_train)) * np.sign(X_train)
        X_test_log  = np.log1p(np.abs(X_test)) * np.sign(X_test)
        
        vt = VarianceThreshold(threshold=0.0)
        X_train_vt = vt.fit_transform(X_train_log)
        X_test_vt = vt.transform(X_test_log)
        
        if X_train_vt.shape[1] == 0:
            continue
            
        scaler = RobustScaler()
        X_train_scaled = scaler.fit_transform(X_train_vt)
        X_test_scaled = scaler.transform(X_test_vt)
        
        k_features = min(8, X_train_scaled.shape[1])
        if k_features == 0:
            continue
            
        selector = SelectKBest(score_func=f_classif, k=k_features)
        X_train_sel = selector.fit_transform(X_train_scaled, y_train)
        X_test_sel = selector.transform(X_test_scaled)
        
        try:
            # SVM configurado para lidar com multiclasse automaticamente (One-vs-Rest / C-Support)
            clf = SVC(C=1.0, kernel='rbf', class_weight='balanced', random_state=seed)
            clf.fit(X_train_sel, y_train)
            pred = clf.predict(X_test_sel)
            
            y_true.extend(y_test)
            y_pred.extend(pred)
        except Exception:
            continue
            
    if len(y_true) > 0:
        resultados_acuracia.append(accuracy_score(y_true, y_pred))
        resultados_acc_balanceada.append(balanced_accuracy_score(y_true, y_pred))

print("\n" + "="*45)
print(" RESULTADOS FINAIS DA VALIDAÇÃO CRUZADA (MULTICLASSE)")
print("="*45)
if resultados_acuracia:
    print(f"Acurácia Média Final:        {np.mean(resultados_acuracia)*100:.2f}% (± {np.std(resultados_acuracia)*100:.2f}%)")
    print(f"Acurácia Balanceada Média:   {np.mean(resultados_acc_balanceada)*100:.2f}% (± {np.std(resultados_acc_balanceada)*100:.2f}%)")
else:
    print("Atenção: Verifique se os dados de treino contêm as classes de emoção distribuídas entre os sujeitos.")
print("="*45)

--- ETAPA 1: Processando os Dados e Mapeando as Emoções ---
Total de arquivos encontrados: 41
Dataset consolidado! Classes alvo encontradas: ['Feliz', 'Nenhuma', 'Neutro', 'Raiva']

--- ETAPA 2: Executando o Pipeline de Machine Learning (Multiclasse) ---

 RESULTADOS FINAIS DA VALIDAÇÃO CRUZADA (MULTICLASSE)
Acurácia Média Final:        53.66% (± 0.00%)
Acurácia Balanceada Média:   53.66% (± 0.00%)


In [14]:
import numpy as np
import pandas as pd
from sklearn.model_selection import LeaveOneOut, StratifiedKFold
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import GradientBoostingClassifier # ou o classificador que você está a usar
from sklearn.metrics import accuracy_score, classification_report

# 1. Configuração dos dados (exemplo estruturado)
# Substitua X e y pelos seus DataFrames/Arrays reais de features e labels
# X = suas features (ex: colunas do BCEA e outras métricas)
# y = suas classes (rótulos de emoção)

print("Iniciando o pipeline otimizado...")

# Definimos apenas 1 seed fixa para garantir determinismo e poupar tempo
SEED = 42
np.random.seed(SEED)

# Inicializadores do Nested CV
outer_cv = LeaveOneOut() # Loop externo: LOOCV (N=41)
# Otimização: Trocamos o RepeatedStratifiedKFold por um StratifiedKFold simples no inner loop
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

y_true = []
y_pred = []

# 2. Loop Externo do Nested LOOCV
for train_idx, test_idx in outer_cv.split(X, y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    # 3. Seleção de Features no Inner Loop (Exemplo com SelectKBest)
    # Selecionamos as melhores features baseadas estritamente nos dados de treino do fold externo
    selector = SelectKBest(score_func=f_classif, k=10) # Ajuste o 'k' conforme necessário
    X_train_selected = selector.fit_transform(X_train, y_train)
    X_test_selected = selector.transform(X_test)
    
    # 4. Treinamento do Classificador
    model = GradientBoostingClassifier(random_state=SEED)
    model.fit(X_train_selected, y_train)
    
    # 5. Predição no Teste Externo (sujeito deixado de fora)
    prediction = model.predict(X_test_selected)
    
    y_true.append(y_test.values[0])
    y_pred.append(prediction[0])

# 6. Avaliação final dos resultados consolidados
final_accuracy = accuracy_score(y_true, y_pred)
print(f"\nAcurácia Global Otimizada: {final_accuracy * 100:.2f}%")
print("\nRelatório de Classificação:")
print(classification_report(y_true, y_pred))

Iniciando o pipeline otimizado...


AttributeError: 'numpy.ndarray' object has no attribute 'iloc'

In [15]:
import numpy as np
import pandas as pd
from sklearn.model_selection import LeaveOneOut, StratifiedKFold
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, balanced_accuracy_score

print("Iniciando o pipeline otimizado...")

SEED = 42
np.random.seed(SEED)

# Certifique-se de que X e y são numpy arrays ou converta-os de forma segura:
# Se X for um DataFrame, podemos usar X.values. Se já for array, fica igual.
X_arr = X.values if hasattr(X, "values") else np.array(X)
y_arr = y.values if hasattr(y, "values") else np.array(y)

outer_cv = LeaveOneOut() 
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

y_true = []
y_pred = []

for train_idx, test_idx in outer_cv.split(X_arr, y_arr):
    # Correção: usando indexação padrão de numpy/arrays em vez de .iloc
    X_train, X_test = X_arr[train_idx], X_arr[test_idx]
    y_train, y_test = y_arr[train_idx], y_arr[test_idx]
    
    # Seleção de Features
    # Nota: Se 'k' for maior que o número total de colunas de X, ajuste para o tamanho máximo ou deixe 'all'
    k_features = min(10, X_train.shape[1])
    selector = SelectKBest(score_func=f_classif, k=k_features)
    X_train_selected = selector.fit_transform(X_train, y_train)
    X_test_selected = selector.transform(X_test)
    
    # Classificador
    model = GradientBoostingClassifier(random_state=SEED)
    model.fit(X_train_selected, y_train)
    
    prediction = model.predict(X_test_selected)
    
    y_true.append(y_test[0])
    y_pred.append(prediction[0])

# Avaliação final
final_accuracy = accuracy_score(y_true, y_pred)
balanced_acc = balanced_accuracy_score(y_true, y_pred)

print(f"\nAcurácia Global Otimizada: {final_accuracy * 100:.2f}%")
print(f"Acurácia Balanceada: {balanced_acc * 100:.2f}%")
print("\nRelatório de Classificação:")
print(classification_report(y_true, y_pred, zero_division=0))

Iniciando o pipeline otimizado...

Acurácia Global Otimizada: 48.17%
Acurácia Balanceada: 48.17%

Relatório de Classificação:
              precision    recall  f1-score   support

       Feliz       0.24      0.27      0.26        41
     Nenhuma       1.00      1.00      1.00        41
      Neutro       0.34      0.34      0.34        41
       Raiva       0.35      0.32      0.33        41

    accuracy                           0.48       164
   macro avg       0.48      0.48      0.48       164
weighted avg       0.48      0.48      0.48       164

